# egy-voice-journal — تشغيل على Colab (مجاني، GPU)

هذا النوتبوك يشغّل خط الأنابيب كامله على سيرفرات Google بدل جهازك — لا ffmpeg ولا Python ولا موديلات تُثبّت أو تُشغَّل محليًا.

**الملف المستهدف في هذا التشغيل:** أطول مكالمة مع نفس المتصل (Ko...)، حاليًا:
`تسجيل المكالمة Ko🧜♀️kվ_٢٦٠٧٠٨_٠٠١٠٠٩.m4a` (~58.9 MB) — غيّره أدناه لو أردت ملفًا آخر.

**قبل التشغيل:**
1. `Runtime` → `Change runtime type` → `T4 GPU` (لو تريد تجربة محرك `qwencleo` المحلي المفتوح؛ Gemini/Groq لا يحتاجان GPU إطلاقًا).
2. جهّز **GitHub Personal Access Token** (repo scope) لأن المستودع خاص — لن يُحفظ على القرص، يُطلب فقط في الذاكرة.
3. جهّز **GEMINI_API_KEY** من https://aistudio.google.com/apikey (وGROQ_API_KEY اختياريًا من https://console.groq.com/keys).

**خصوصية:** لا شيء يُرفع لأي مكان إلا الصوت نفسه إلى المزوّد الذي تختاره (Gemini/Groq/ElevenLabs) — راجع تحذيرات الخصوصية في `README.md` قبل الاختيار.

In [ ]:
# فحص GPU (اختياري -- فقط لو ستجرب qwencleo أو local-whisper)
!nvidia-smi || echo 'لا GPU متاح في هذا الـ runtime -- لا مشكلة لو ستستخدم gemini أو groq فقط'

In [ ]:
# استنساخ المستودع الخاص -- التوكن يُطلب هنا فقط ولا يُحفظ في أي ملف
import getpass, os

GH_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')
REPO = 'eltaweelactuary/egy-voice-journal'

!rm -rf egy-voice-journal
!git clone --quiet https://{GH_TOKEN}@github.com/{REPO}.git
del GH_TOKEN  # يُمحى من الذاكرة فورًا بعد الاستخدام
%cd egy-voice-journal

In [ ]:
# التبعيات الأساسية -- تكفي لكل المحركات القائمة على API (gemini, groq, elevenlabs)
!pip install -q -r requirements.txt

# ffmpeg مثبت مسبقًا على Colab افتراضيًا -- تأكيد فقط
!ffmpeg -version | head -1

In [ ]:
# (اختياري) تفعيل qwencleo -- المحرك الافتراضي في المشروع، يحتاج GPU ليكون عمليًا
# شغّل هذا الخلية فقط لو فعّلت T4 GPU أعلاه وتريد تجربته
RUN_QWENCLEO = False  # غيّرها إلى True لو تريد تجربته

if RUN_QWENCLEO:
    !pip install -q torch torchaudio
    !pip install -q qwencleo-asr --no-deps
    !pip install -q "qwen-asr>=0.0.6" numpy soundfile huggingface_hub

In [ ]:
# مفاتيح المزوّدين -- تُطلب هنا فقط، تعيش في متغيرات البيئة لهذا الـ runtime، لا تُكتب في أي ملف
import getpass, os

os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')
groq_key = getpass.getpass('GROQ_API_KEY (اختياري -- Enter للتخطي): ')
if groq_key.strip():
    os.environ['GROQ_API_KEY'] = groq_key
del groq_key

In [ ]:
# رفع ملف الصوت من جهازك -- الرفع فقط، لا معالجة تحدث على جهازك
from google.colab import files
import shutil, pathlib

pathlib.Path('inbox').mkdir(exist_ok=True)
uploaded = files.upload()  # اختر: تسجيل المكالمة Ko🧜♀️kվ_٢٦٠٧٠٨_٠٠١٠٠٩.m4a

AUDIO_NAME = list(uploaded.keys())[0]
shutil.move(AUDIO_NAME, f'inbox/{AUDIO_NAME}')
AUDIO_PATH = f'inbox/{AUDIO_NAME}'
print('تم رفع:', AUDIO_PATH)

## خطوة ١ -- مقارنة سريعة على عيّنة ٣ دقائق (توصية HANDOFF.md قبل أي تشغيل كامل)

In [ ]:
providers = 'gemini,groq' if os.environ.get('GROQ_API_KEY') else 'gemini'
if RUN_QWENCLEO:
    providers += ',qwencleo'

!python compare.py "{AUDIO_PATH}" --minutes 3 --speakers 2 --providers {providers}

In [ ]:
# اقرأ المقارنة قبل الاستمرار
print(open('comparison.md', encoding='utf-8').read())

## خطوة ٢ -- التفريغ الكامل للملف

غيّر `--provider` أدناه للمحرك الذي أثبت أنه الأفضل من جدول المقارنة أعلاه.
`--speakers 2` لأنها مكالمة بين متحدثين.

In [ ]:
!python transcribe.py "{AUDIO_PATH}" --provider gemini --speakers 2 --out journal

In [ ]:
# اعرض التفريغ هنا مباشرة للمراجعة السريعة
import glob, os
latest_md = max(glob.glob('journal/*.md'), key=os.path.getmtime)
print(open(latest_md, encoding='utf-8').read())

In [ ]:
# تحميل كل مخرجات journal/ (md + txt + srt + json) إلى جهازك -- لا شيء يبقى على سيرفر Colab بعد إغلاق الجلسة
import shutil
shutil.make_archive('journal_output', 'zip', 'journal')
from google.colab import files as files2
files2.download('journal_output.zip')

## ملاحظات

- **لا يُدفع أي شيء إلى المستودع تلقائيًا.** التفريغ فيه محتوى مكالمة شخصية، وقرار رفعه لـ Git أو الاحتفاظ به محليًا فقط يرجع للمالك (راجع القسم ٧ في `HANDOFF.md`).
- إغلاق الـ runtime (`Runtime` → `Disconnect and delete runtime`) يمحو كل شيء: الملف المرفوع، المفاتيح المؤقتة، والمخرجات غير المُنزَّلة.
- لو ظهر خطأ `GEMINI_API_KEY غير موجود` تأكد إنك نفّذت خلية المفاتيح فوق **بعد** الدخول لمجلد المستودع (`%cd egy-voice-journal`).